# Fine-tune WangchanBERTa สำหรับวิเคราะห์ Sentiment ภาษาไทย
**โปรเจกต์:** AI Customer Sentiment & Auto-Draft Reply (IT232)

Notebook นี้ "สอน" โมเดลภาษาไทย **WangchanBERTa** ให้แยกความคิดเห็นเป็น **Positive / Negative** (ตาม IT Workflow ในโจทย์)
แล้วเทียบผลกับ 3 แบบ เพื่อเอาตัวเลขไปใช้ในสไลด์

| แบบ | คืออะไร |
|---|---|
| ก่อนจูน | WangchanBERTa ที่ยังไม่ได้สอน (หัวจำแนกสุ่ม) |
| Baseline | TF-IDF + Logistic Regression (Machine Learning แบบดั้งเดิม) |
| **หลังจูน** | WangchanBERTa ที่ fine-tune แล้ว |
| LLM + Prompt | โมเดลเดียวกับที่ใช้ในแอป (ไม่บังคับ ต้องใส่ API key) |

**ข้อมูล:** Wisesight Sentiment Corpus (ข้อความโซเชียลมีเดียภาษาไทยที่ติดป้ายแล้ว, ใบอนุญาต CC0) จาก
[github.com/PyThaiNLP/wisesight-sentiment](https://github.com/PyThaiNLP/wisesight-sentiment) ใช้เฉพาะป้าย pos/neg
และ (ไม่บังคับ) รีวิวไทยที่กลุ่มเก็บและติดป้ายเอง

## วิธีใช้
1. เมนู **Runtime → Change runtime type → T4 GPU** → Save
2. ตั้งค่าในเซลล์ "ตั้งค่า" ด้านล่าง (ค่าเริ่มต้นใช้ได้เลย)
3. เมนู **Runtime → Run all** ใช้เวลาประมาณ 15–30 นาที
4. (ไม่บังคับ) ถ้าอยากเทียบกับ LLM ในแอป ให้กรอกช่อง ENDPOINT / MODEL_ID / API_KEY ในเซลล์ "เชื่อมต่อ LLM" ก่อนกด Run all ถ้าเว้นว่างจะข้ามขั้นนี้

In [ ]:
# ติดตั้งไลบรารี (Colab มี transformers/torch อยู่แล้ว)
!pip -q install transformers accelerate sentencepiece datasets scikit-learn

## ตั้งค่า

In [ ]:
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"
MAX_TRAIN = 8000          # จำนวนข้อความที่ใช้เทรน (None = ใช้ทั้งหมด ~9,400 ช้าขึ้น)
EPOCHS = 2
MAX_LEN = 128
LR = 3e-5
BATCH = 32
SEED = 42

USE_OWN_DATA = False      # True = อัปโหลดไฟล์รีวิวไทยที่กลุ่มติดป้ายเอง (คอลัมน์ text, label)
ADD_OWN_TO_TRAIN = True   # True = เอาครึ่งหนึ่งของข้อมูลกลุ่มไปช่วยเทรน อีกครึ่งใช้ทดสอบ

N_LLM = 200               # จำนวนข้อความที่ใช้เทียบ (เรียก API เท่านี้ครั้ง)

### 🔑 (ไม่บังคับ) เชื่อมต่อ LLM เพื่อเทียบผล
กรอกในช่องด้านล่าง (ใช้ค่าเดียวกับในแอป) **ถ้าไม่อยากเทียบ เว้นว่างทั้งหมด** แล้วกด Run all ได้เลย

⚠️ API key จะถูกบันทึกไปกับ Notebook ถ้ากด Save ก่อนแชร์ให้ลบช่อง API key ให้ว่างก่อน

In [ ]:
#@title เชื่อมต่อ LLM (เว้นว่าง = ข้ามการเทียบ)
ENDPOINT = ""  #@param {type:"string"}
MODEL_ID = ""  #@param {type:"string"}
API_KEY = ""   #@param {type:"string"}

## 1) โหลดข้อมูล Wisesight

In [ ]:
import re, random, urllib.request
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

random.seed(SEED); np.random.seed(SEED)
LABELS = ["negative", "positive"]
L2I = {l: i for i, l in enumerate(LABELS)}
BASE = "https://raw.githubusercontent.com/PyThaiNLP/wisesight-sentiment/master/kaggle-competition/"

def _lines(name):
    raw = urllib.request.urlopen(BASE + name, timeout=60).read().decode("utf-8")
    return [x.rstrip("\r") for x in raw.split("\n")]

def load_split(split):
    texts, labels = _lines(f"{split}.txt"), _lines(f"{split}_label.txt")
    n = min(len(texts), len(labels))
    df = pd.DataFrame({"text": texts[:n], "label": labels[:n]})
    df = df[df["label"].isin(["pos", "neg"]) & (df["text"].str.strip() != "")]
    df["label"] = df["label"].map({"neg": "negative", "pos": "positive"})
    return df.reset_index(drop=True)

train_all = load_split("train")
test_df = load_split("test")
train_df, val_df = train_test_split(train_all, test_size=0.1, stratify=train_all["label"], random_state=SEED)
if MAX_TRAIN and len(train_df) > MAX_TRAIN:
    train_df, _ = train_test_split(train_df, train_size=MAX_TRAIN, stratify=train_df["label"], random_state=SEED)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)

print("train:", train_df["label"].value_counts().to_dict())
print("val  :", val_df["label"].value_counts().to_dict())
print("test :", test_df["label"].value_counts().to_dict())
test_df.sample(5, random_state=SEED)

## 2) (ไม่บังคับ) ข้อมูลรีวิวที่กลุ่มเก็บเอง

In [ ]:
own_test = None
if USE_OWN_DATA:
    import io
    from google.colab import files
    up = files.upload()
    name, data = next(iter(up.items()))
    sep = "\t" if name.lower().endswith(".tsv") else ","
    own = pd.read_csv(io.BytesIO(data), sep=sep)
    cols = {c.strip().lower(): c for c in own.columns}
    tcol = next(cols[c] for c in ["text", "review", "comment", "รีวิว"] if c in cols)
    lcol = next(cols[c] for c in ["label", "liked", "sentiment"] if c in cols)
    own = pd.DataFrame({"text": own[tcol].astype(str), "label": own[lcol].astype(str).str.strip().str.lower()})
    own["label"] = own["label"].replace({"1": "positive", "0": "negative", "pos": "positive", "neg": "negative"})
    own = own[own["label"].isin(LABELS)].reset_index(drop=True)   # ตัด neutral ออก (โจทย์ใช้ Positive/Negative)
    print("ข้อมูลกลุ่ม:", own["label"].value_counts().to_dict())
    if ADD_OWN_TO_TRAIN and len(own) >= 20:
        own_train, own_test = train_test_split(own, test_size=0.5, stratify=own["label"], random_state=SEED)
        train_df = pd.concat([train_df, own_train], ignore_index=True)
        print("เพิ่มเข้าชุดเทรน", len(own_train), "| เก็บไว้ทดสอบ", len(own_test))
    else:
        own_test = own
else:
    print("ข้าม (USE_OWN_DATA = False)")

## 3) Baseline: TF-IDF + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def scores(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred), "macro_f1": f1_score(y_true, y_pred, average="macro")}

# ภาษาไทยไม่มีเว้นวรรคระหว่างคำ จึงใช้ n-gram ระดับตัวอักษร
baseline = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=2000, class_weight="balanced"))
baseline.fit(train_df["text"], train_df["label"])
base_pred = baseline.predict(test_df["text"])
results = {"Baseline (TF-IDF + LR)": scores(test_df["label"], base_pred)}
print(results)

## 4) เตรียม WangchanBERTa

In [ ]:
import inspect, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี (จะช้ามาก ให้เปลี่ยน Runtime เป็น T4 GPU)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def prep(t):
    # กติกาเตรียมข้อความของ WangchanBERTa: ตัวพิมพ์เล็ก, เว้นวรรคแทนด้วย <_>
    t = re.sub(r"\s+", " ", str(t).lower()).strip()
    return t.replace(" ", "<_>")

def to_ds(df):
    ds = Dataset.from_dict({"text": [prep(t) for t in df["text"]], "label": [L2I[l] for l in df["label"]]})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=["text"])

train_ds, val_ds, test_ds = to_ds(train_df), to_ds(val_df), to_ds(test_df)

def new_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=L2I)

def compute_metrics(p):
    pred = np.argmax(p.predictions, axis=-1)
    return {"accuracy": accuracy_score(p.label_ids, pred), "macro_f1": f1_score(p.label_ids, pred, average="macro")}

def make_args(**kw):
    # รองรับ transformers หลายเวอร์ชัน (evaluation_strategy เปลี่ยนชื่อเป็น eval_strategy)
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" not in params and "eval_strategy" in kw:
        kw["evaluation_strategy"] = kw.pop("eval_strategy")
    return TrainingArguments(**{k: v for k, v in kw.items() if k in params or k == "evaluation_strategy"})

def make_trainer(model, args, **kw):
    params = inspect.signature(Trainer.__init__).parameters
    tok_kw = {"processing_class": tokenizer} if "processing_class" in params else {"tokenizer": tokenizer}
    return Trainer(model=model, args=args, data_collator=DataCollatorWithPadding(tokenizer),
                   compute_metrics=compute_metrics, **tok_kw, **kw)

## 5) ก่อนจูน: โมเดลที่ยังไม่ได้สอน

In [ ]:
eval_args = make_args(output_dir="tmp_eval", per_device_eval_batch_size=64, report_to="none")
before = make_trainer(new_model(), eval_args).predict(test_ds)
results["WangchanBERTa ก่อนจูน"] = {"accuracy": before.metrics["test_accuracy"], "macro_f1": before.metrics["test_macro_f1"]}
print(results["WangchanBERTa ก่อนจูน"])

## 6) Fine-tune

In [ ]:
import time
model = new_model()
args = make_args(
    output_dir="wcb_sentiment", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
    weight_decay=0.01, warmup_ratio=0.1, eval_strategy="epoch", save_strategy="no",
    logging_steps=50, fp16=torch.cuda.is_available(), seed=SEED, report_to="none")
trainer = make_trainer(model, args, train_dataset=train_ds, eval_dataset=val_ds)
t0 = time.time()
trainer.train()
train_minutes = (time.time() - t0) / 60
print(f"เทรนเสร็จใน {train_minutes:.1f} นาที")

## 7) หลังจูน: ทดสอบกับชุดทดสอบ (ไม่เคยเห็นตอนเทรน)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

after = trainer.predict(test_ds)
wcb_pred = [LABELS[i] for i in np.argmax(after.predictions, axis=-1)]
results["WangchanBERTa หลังจูน"] = scores(test_df["label"], wcb_pred)
print(classification_report(test_df["label"], wcb_pred, digits=3))
ConfusionMatrixDisplay(confusion_matrix(test_df["label"], wcb_pred, labels=LABELS), display_labels=LABELS).plot(cmap="Blues")
plt.title("Fine-tuned WangchanBERTa (test set)"); plt.show()

## 8) (ไม่บังคับ) เทียบกับ LLM ที่ใช้ในแอป
ใช้ค่าที่กรอกในช่อง ENDPOINT / MODEL_ID / API_KEY ด้านบน (ถ้าเว้นว่างจะข้ามขั้นนี้)

In [ ]:
import json, getpass, requests
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urlparse

llm_sample = pd.concat([g.sample(min(len(g), N_LLM // 2), random_state=SEED)
                        for _, g in test_df.groupby("label")]).reset_index(drop=True)

def normalize_endpoint(url):
    url = (url or "").strip().rstrip("/")
    if not re.match(r"^https?://", url): url = "https://" + url
    if urlparse(url).path in ("", "/"): url += "/v1"
    if not url.endswith("/chat/completions"): url += "/chat/completions"
    return url

SYS = ("คุณคือระบบจำแนกความคิดเห็นลูกค้าภาษาไทย ตอบคำเดียวเท่านั้น: positive หรือ negative\n"
       "ตัวอย่าง: 'ของดีมาก ส่งไว' -> positive | 'กล่องบุบ ของแตก' -> negative")

def llm_label(text, cfg):
    for _ in range(3):
        try:
            r = requests.post(cfg["url"], timeout=60, headers={"Authorization": f"Bearer {cfg['key']}"},
                              json={"model": cfg["model"], "temperature": 0,
                                    "messages": [{"role": "system", "content": SYS}, {"role": "user", "content": text}]})
            if r.status_code == 429 or r.status_code >= 500:
                time.sleep(3); continue
            out = r.json()["choices"][0]["message"]["content"].lower()
            return "positive" if "positive" in out and "negative" not in out else "negative" if "negative" in out else None
        except Exception:
            time.sleep(2)
    return None

llm_pred = None
if ENDPOINT.strip():
    key = API_KEY.strip() or getpass.getpass("API key (ช่องด้านบนว่าง จึงถามตรงนี้): ")
    cfg = {"url": normalize_endpoint(ENDPOINT), "model": MODEL_ID.strip(), "key": key}
    with ThreadPoolExecutor(4) as ex:
        llm_pred = list(ex.map(lambda t: llm_label(t, cfg), llm_sample["text"]))
    ok = [p is not None for p in llm_pred]
    print(f"LLM ตอบสำเร็จ {sum(ok)}/{len(ok)}")
else:
    print("ข้ามการเทียบกับ LLM (ไม่ได้กรอก ENDPOINT)")

## 9) สรุปผล (ตารางนี้ใช้ใส่สไลด์ได้เลย)

In [ ]:
table = pd.DataFrame(results).T
table.index.name = f"ชุดทดสอบ Wisesight {len(test_df)} ข้อความ"
display((table * 100).round(1).rename(columns={"accuracy": "Accuracy (%)", "macro_f1": "Macro-F1 (%)"}))

if llm_pred is not None:
    m = [p is not None for p in llm_pred]
    sub = llm_sample[m]
    comp = {
        "Baseline (TF-IDF + LR)": scores(sub["label"], baseline.predict(sub["text"])),
        "WangchanBERTa หลังจูน": scores(sub["label"], [LABELS[i] for i in np.argmax(trainer.predict(to_ds(sub)).predictions, -1)]),
        "LLM + Prompt": scores(sub["label"], [p for p in llm_pred if p is not None]),
    }
    comp_t = (pd.DataFrame(comp).T * 100).round(1).rename(columns={"accuracy": "Accuracy (%)", "macro_f1": "Macro-F1 (%)"})
    comp_t.index.name = f"ชุดเดียวกัน {len(sub)} ข้อความ"
    display(comp_t)

names = {"WangchanBERTa ก่อนจูน": "WCB before", "Baseline (TF-IDF + LR)": "TF-IDF+LR", "WangchanBERTa หลังจูน": "WCB fine-tuned"}
order = [k for k in names if k in results]
plt.figure(figsize=(6, 3.2))
bars = plt.bar([names[k] for k in order], [results[k]["accuracy"] * 100 for k in order], color=["#bbb", "#8aa", "#2a6fdb"])
plt.bar_label(bars, fmt="%.1f%%"); plt.ylim(0, 100); plt.ylabel("Accuracy (%)"); plt.title("Sentiment accuracy on test set")
plt.tight_layout(); plt.savefig("accuracy_chart.png", dpi=200); plt.show()
table.to_csv("results.csv")
print(f"เทรน {len(train_df)} ข้อความ, {EPOCHS} epochs, {train_minutes:.1f} นาที บน {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 10) (ถ้าอัปโหลดข้อมูลกลุ่ม) ทดสอบกับรีวิวร้านค้าจริง

In [ ]:
if own_test is not None and len(own_test):
    p = [LABELS[i] for i in np.argmax(trainer.predict(to_ds(own_test)).predictions, -1)]
    print("WangchanBERTa หลังจูน กับรีวิวของกลุ่ม:", scores(own_test["label"], p))
    print("Baseline กับรีวิวของกลุ่ม:", scores(own_test["label"], baseline.predict(own_test["text"])))
else:
    print("ไม่มีข้อมูลกลุ่ม (ข้าม)")

## 11) ลองพิมพ์เอง

In [ ]:
def predict(texts):
    enc = tokenizer([prep(t) for t in texts], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        prob = torch.softmax(model(**enc).logits, -1).cpu().numpy()
    return [(t, LABELS[int(p.argmax())], round(float(p.max()), 3)) for t, p in zip(texts, prob)]

for row in predict(["ของดีมาก ส่งไว แพ็คแน่น ประทับใจ", "สั่งไปสองอาทิตย์ยังไม่ได้ของ แย่มาก",
                    "กล่องบุบ ของข้างในแตก", "ใช้แล้วผิวดีขึ้นจริง จะซื้อซ้ำ"]):
    print(row)

## 12) (ไม่บังคับ) บันทึกโมเดลลง Google Drive

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    out = "/content/drive/MyDrive/wangchanberta_sentiment"
    trainer.save_model(out); tokenizer.save_pretrained(out)
    print("บันทึกที่", out)